<a href="https://colab.research.google.com/github/RAndersen1/risky.nvim/blob/master/10_Dimentional_Slide_Model_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.7 MB/s eta 0:00:00


In [2]:
import json, math, os, random
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import trange
from sklearn.cluster import KMeans
from torch_geometric.nn import GATConv
from torch_geometric.data import Data
import pathlib

# —— paths ————————————————————————————————————————————————
DATA_PATH  = Path('/content/embeddings_normalized.jsonl')
STATS_PATH = Path('/content/embeddings_stats.json')

# —— Hyper-params ————————————————————————————————
PAD_VALUE   = -5.0
K           = 3
MAX_TOKENS  = 50
EMBEDDING_DIM = 10  # 6D geometry + 4D color
D_MODEL     = 128
N_HEADS     = 8
LAYERS_CTX  = 2
BATCH_SIZE  = 8
EPOCHS      = 200
LR          = 5e-5
CLIP_GRAD   = .25
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'
SLIDE_WIDTH_EMU  = 9_144_000
SLIDE_HEIGHT_EMU = 6_858_000
NUM_LAYOUT_TYPES = 10
NUM_PRESENTATIONS = 46

torch.manual_seed(0); random.seed(0); np.random.seed(0)

# Load stats (mean/std for the first 4 geometric features)
with open(STATS_PATH) as f:
    stats = json.load(f)
mu, std = stats['mean'], stats['std']
if len(mu) != 4 or len(std) != 4:
    raise ValueError(f"Expected 4 values for mu/std, got {len(mu)}/{len(std)}")

# ======================================================================
# 1. Load Data
# ======================================================================
def load_slides(jsonl_path):
    mats, masks, pres_ids = [], [], []
    with open(jsonl_path) as f:
        for line in f:
            data = json.loads(line)
            emb = torch.tensor(data['embedding'], dtype=torch.float32)  # 50x10 embedding
            msk = (emb[:, :6] != PAD_VALUE).any(-1).float()  # Mask based on geometry only
            pres_id = data['presentation_id']
            if pres_id >= NUM_PRESENTATIONS or pres_id < 0:
                raise ValueError(f"Invalid presentation_id: {pres_id}")
            mats.append(emb)
            masks.append(msk)
            pres_ids.append(pres_id)
    print(f"Loaded pres_ids range: min={min(pres_ids)}, max={max(pres_ids)}")
    return mats, masks, pres_ids

all_mats, all_masks, all_pres_ids = load_slides(DATA_PATH)
train_ds_raw = (all_mats[:int(len(all_mats)*0.8)], all_masks[:int(len(all_mats)*0.8)], all_pres_ids[:int(len(all_mats)*0.8)])
val_ds_raw = (all_mats[int(len(all_mats)*0.8):], all_masks[int(len(all_mats)*0.8):], all_pres_ids[int(len(all_mats)*0.8):])

# Cluster slides into layout types using geometry only
geometry_embeddings = torch.stack(all_mats)[:, :, :6].reshape(len(all_mats), -1).numpy()
kmeans = KMeans(n_clusters=NUM_LAYOUT_TYPES, n_init=10, random_state=0).fit(geometry_embeddings)
layout_labels = kmeans.labels_
print(f"layout_labels range: min={layout_labels.min()}, max={layout_labels.max()}")

# ======================================================================
# 2. Dataset & Loaders
# ======================================================================
class SeqDataset(Dataset):
    def __init__(self, mats, masks, pres_ids, labels, K):
        self.mats = mats
        self.masks = masks
        self.pres_ids = pres_ids
        self.labels = labels
        self.K = K

    def __len__(self):
        return len(self.mats) - self.K

    def __getitem__(self, idx):
        ctx_emb = torch.stack(self.mats[idx:idx+self.K])      # (K, 50, 10)
        ctx_m = torch.stack(self.masks[idx:idx+self.K])       # (K, 50)
        ctx_p = [min(max(p, 0), NUM_PRESENTATIONS-1) for p in self.pres_ids[idx:idx+self.K]]
        tgt_emb = self.mats[idx+self.K]                       # (50, 10)
        tgt_m = self.masks[idx+self.K]                        # (50,)
        tgt_p = min(max(self.pres_ids[idx+self.K], 0), NUM_PRESENTATIONS-1)
        tgt_l = self.labels[idx+self.K]                       # Layout label
        return ctx_emb, ctx_m, ctx_p, tgt_emb, tgt_m, tgt_p, tgt_l

def collate(batch):
    ctx_emb, ctx_m, ctx_p, tgt_emb, tgt_m, tgt_p, tgt_l = zip(*batch)
    ctx_emb = torch.stack(ctx_emb)                        # (batch_size, K, 50, 10)
    ctx_m = torch.stack(ctx_m)                            # (batch_size, K, 50)
    ctx_p = torch.tensor([p for p in ctx_p], dtype=torch.long)  # (batch_size, K)
    tgt_emb = torch.stack(tgt_emb)                        # (batch_size, 50, 10)
    tgt_m = torch.stack(tgt_m)                            # (batch_size, 50)
    tgt_p = torch.tensor(tgt_p, dtype=torch.long)         # (batch_size,)
    tgt_l = torch.tensor(tgt_l, dtype=torch.long)         # (batch_size,)
    return ctx_emb, ctx_m, ctx_p, tgt_emb, tgt_m, tgt_p, tgt_l

train_loader = DataLoader(
    SeqDataset(*train_ds_raw, layout_labels[:int(len(all_mats)*0.8)], K),
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate
)
val_loader = DataLoader(
    SeqDataset(*val_ds_raw, layout_labels[int(len(all_mats)*0.8):], K),
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate
)

# ======================================================================
# 3. Model
# ======================================================================
def create_graph(emb, msk, threshold=1.0):
    """Create a graph from slide embedding where nodes are shapes and edges connect nearby shapes."""
    emb = emb.to(DEVICE)
    msk = msk.to(DEVICE)
    valid = msk == 1
    features = emb[valid, :4]  # (N, 4) - [x, y, width, height]
    if features.size(0) < 2:
        return Data(
            x=features if features.size(0) > 0 else torch.zeros(1, 4, device=DEVICE),
            edge_index=torch.empty(2, 0, dtype=torch.long, device=DEVICE)
        )
    centers = features[:, :2] + features[:, 2:] / 2  # (N, 2) - Center points
    dist = torch.cdist(centers, centers)             # (N, N) - Pairwise distances
    edge_index = (dist < threshold).nonzero(as_tuple=False).t()  # (2, E)
    if edge_index.max() >= features.size(0):
        raise ValueError(f"edge_index out of bounds: max={edge_index.max()}, num_nodes={features.size(0)}")
    return Data(x=features, edge_index=edge_index)

class SlideGNNEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.node_proj = nn.Linear(4, D_MODEL)
        self.gat1 = GATConv(D_MODEL, D_MODEL // N_HEADS, heads=N_HEADS)
        self.gat2 = GATConv(D_MODEL, D_MODEL // N_HEADS, heads=N_HEADS)
        self.global_pool = nn.Linear(D_MODEL, D_MODEL)

    def forward(self, emb, msk):
        graph = create_graph(emb, msk)
        x = self.node_proj(graph.x)                  # Project node features
        x = F.relu(self.gat1(x, graph.edge_index))   # First GAT layer
        x = F.relu(self.gat2(x, graph.edge_index))   # Second GAT layer
        pooled = self.global_pool(x.mean(dim=0))     # Global pooling
        return pooled

class ContextEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.pos_emb = nn.Parameter(torch.randn(K, D_MODEL))
        enc_layer = nn.TransformerEncoderLayer(D_MODEL, N_HEADS, 4*D_MODEL, batch_first=True)
        self.enc = nn.TransformerEncoder(enc_layer, LAYERS_CTX)
        self.pres_emb = nn.Embedding(NUM_PRESENTATIONS, D_MODEL)

    def forward(self, slides, pres_ids):
        x = slides + self.pos_emb + self.pres_emb(pres_ids)  # Combine embeddings
        return self.enc(x)[:, -1]                            # Last context vector

class NextSlideModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc_slide = SlideGNNEncoder()
        self.enc_ctx = ContextEncoder()
        self.layout_classifier = nn.Linear(D_MODEL, NUM_LAYOUT_TYPES)

    def forward(self, ctx_emb, ctx_m, ctx_p):
        batch_size, K, _, _ = ctx_emb.shape
        if ctx_p.max() >= NUM_PRESENTATIONS or ctx_p.min() < 0:
            raise ValueError(f"ctx_p out of bounds: min={ctx_p.min()}, max={ctx_p.max()}")
        slide_vecs = []
        for b in range(batch_size):
            for k in range(K):
                emb = ctx_emb[b, k]    # (50, 10)
                msk = ctx_m[b, k]      # (50,)
                vec = self.enc_slide(emb, msk)
                slide_vecs.append(vec)
        slide_vecs = torch.stack(slide_vecs).view(batch_size, K, D_MODEL)
        ctx_vec = self.enc_ctx(slide_vecs, ctx_p)
        layout_logits = self.layout_classifier(ctx_vec)
        return layout_logits

# ======================================================================
# 4. Losses
# ======================================================================
ce_loss = nn.CrossEntropyLoss()

# ======================================================================
# 5. Training
# ======================================================================
model = NextSlideModel().to(DEVICE)
opt = torch.optim.Adam(model.parameters(), lr=LR)

for ep in trange(1, EPOCHS+1, desc='epochs'):
    model.train()
    run_loss = 0.0
    for ctx_emb, ctx_m, ctx_p, tgt_emb, tgt_m, tgt_p, tgt_l in train_loader:
        ctx_emb = ctx_emb.to(DEVICE)
        ctx_m = ctx_m.to(DEVICE)
        ctx_p = ctx_p.to(DEVICE)
        tgt_l = tgt_l.to(DEVICE)
        if ctx_p.max() >= NUM_PRESENTATIONS or ctx_p.min() < 0:
            raise ValueError(f"Train ctx_p out of bounds: {ctx_p}")
        if tgt_l.max() >= NUM_LAYOUT_TYPES or tgt_l.min() < 0:
            raise ValueError(f"Train tgt_l out of bounds: {tgt_l}")
        p_layout = model(ctx_emb, ctx_m, ctx_p)
        loss = ce_loss(p_layout, tgt_l)
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        opt.step()
        run_loss += loss.item()
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for ctx_emb, ctx_m, ctx_p, tgt_emb, tgt_m, tgt_p, tgt_l in val_loader:
            ctx_emb = ctx_emb.to(DEVICE)
            ctx_m = ctx_m.to(DEVICE)
            ctx_p = ctx_p.to(DEVICE)
            tgt_l = tgt_l.to(DEVICE)
            if ctx_p.max() >= NUM_PRESENTATIONS or ctx_p.min() < 0:
                raise ValueError(f"Val ctx_p out of bounds: {ctx_p}")
            if tgt_l.max() >= NUM_LAYOUT_TYPES or tgt_l.min() < 0:
                raise ValueError(f"Val tgt_l out of bounds: {tgt_l}")
            p_layout = model(ctx_emb, ctx_m, ctx_p)
            val_loss += ce_loss(p_layout, tgt_l).item()
    if ep % 25 == 0 or ep == 1:
        print(f"E{ep:03d} | train {run_loss/len(train_loader):.4f} | val {val_loss/len(val_loader):.4f}")

torch.save(model.state_dict(), "next_slide_with_color.pt")
print("✅ Model saved")

import torch
from pathlib import Path
import json
import random

# Assuming these are defined elsewhere in your code
# SLIDE_WIDTH_EMU = some_integer_value  # e.g., 9144000 (PowerPoint EMU units for slide width)
# SLIDE_HEIGHT_EMU = some_integer_value # e.g., 6858000 (PowerPoint EMU units for slide height)
# std = list_of_standard_deviations     # e.g., [std_x, std_y, std_w, std_h]
# mu = list_of_means                    # e.g., [mu_x, mu_y, mu_w, mu_h]
# K = number_of_context_slides          # e.g., 3
# PAD_VALUE = some_padding_value        # e.g., -1

def compute_average_color(emb, msk, device='cuda'):
    """Compute the average color of an embedding tensor, ensuring device consistency."""
    emb = emb.to(device)
    msk = msk.to(device)
    valid = msk == 1
    if valid.sum() == 0:
        return torch.tensor([0.0, 0.0, 0.0, 0.0], device=device)  # RGBA
    colors = emb[valid, 6:]  # Assuming color in dimensions 6-9 (RGBA)
    avg_color = colors.mean(dim=0)
    return avg_color

@torch.no_grad()
def generate_next_slide(model, ctx_emb, ctx_m, ctx_p, all_mats, layout_labels, device='cuda'):
    """Generate the next slide, ensuring all tensors are on the specified device."""
    model.eval()
    ctx_emb = ctx_emb.to(device)
    ctx_m = ctx_m.to(device)
    ctx_p = torch.tensor(ctx_p, dtype=torch.long).to(device).unsqueeze(0)  # (1, K)
    p_layout = model(ctx_emb.unsqueeze(0), ctx_m.unsqueeze(0), ctx_p)
    layout_type = torch.argmax(p_layout, dim=-1).item()
    candidates = [(m, i) for i, (m, l) in enumerate(zip(all_mats, layout_labels)) if l == layout_type]

    if not candidates:
        gen = all_mats[0].to(device)
    else:
        # Compute average color of context slides
        ctx_avg_colors = []
        for k in range(K):
            emb = ctx_emb[k]  # (50, 10)
            msk = ctx_m[k]    # (50,)
            avg_color = compute_average_color(emb, msk, device=device)
            ctx_avg_colors.append(avg_color)
        ctx_avg_color = torch.stack(ctx_avg_colors).mean(dim=0)

        # Select candidate with closest average color
        min_dist = float('inf')
        best_candidate = None
        for cand_emb, idx in candidates:
            cand_emb = cand_emb.to(device)
            cand_msk = (cand_emb[:, :6] != PAD_VALUE).any(-1).float().to(device)
            cand_avg_color = compute_average_color(cand_emb, cand_msk, device=device)
            dist = torch.norm(cand_avg_color - ctx_avg_color)
            if dist < min_dist:
                min_dist = dist
                best_candidate = cand_emb
        gen = best_candidate if best_candidate is not None else random.choice(candidates)[0].to(device)

    # Denormalize and clamp geometry
    slide_width_emu = torch.tensor(SLIDE_WIDTH_EMU, device=device, dtype=torch.float32)
    slide_height_emu = torch.tensor(SLIDE_HEIGHT_EMU, device=device, dtype=torch.float32)
    gen_denorm = gen[:, :4] * torch.tensor(std[:4], device=device) + torch.tensor(mu[:4], device=device)

    # Clamp x and y coordinates (scalars for min and max)
    gen_denorm[:, 0] = torch.clamp(gen_denorm[:, 0], 0, SLIDE_WIDTH_EMU)
    gen_denorm[:, 1] = torch.clamp(gen_denorm[:, 1], 0, SLIDE_HEIGHT_EMU)

    # Clamp width and height using torch.max and torch.min
    zero_tensor = torch.tensor(0.0, device=device, dtype=torch.float32)
    gen_denorm[:, 2] = torch.max(gen_denorm[:, 2], zero_tensor)  # Ensure width >= 0
    gen_denorm[:, 2] = torch.min(gen_denorm[:, 2], slide_width_emu - gen_denorm[:, 0])  # Ensure width <= slide_width - x
    gen_denorm[:, 3] = torch.max(gen_denorm[:, 3], zero_tensor)  # Ensure height >= 0
    gen_denorm[:, 3] = torch.min(gen_denorm[:, 3], slide_height_emu - gen_denorm[:, 1])  # Ensure height <= slide_height - y

    # Renormalize the clamped geometry
    gen[:, :4] = (gen_denorm - torch.tensor(mu[:4], device=device)) / torch.tensor(std[:4], device=device)

    return gen

# Example usage (ensure val_ds_raw, all_mats, layout_labels, and model are defined)
ctx_emb = torch.stack(val_ds_raw[0][:K])  # (K, 50, 10)
ctx_m = torch.stack(val_ds_raw[1][:K])    # (K, 50)
ctx_p = val_ds_raw[2][:K]                 # List of K pres_ids
gen = generate_next_slide(model, ctx_emb, ctx_m, ctx_p, all_mats, layout_labels, device='cuda')
print("Generated slide shape:", gen.shape)

out_path = Path('generated_slide_with_color.json')
json.dump({"generated": gen.cpu().tolist()}, out_path.open("w"), indent=2)
print("✅ Slide JSON saved:", out_path)

Loaded pres_ids range: min=0, max=45
layout_labels range: min=0, max=9


epochs:   0%|          | 0/200 [00:00<?, ?it/s]

E001 | train 2.3183 | val 2.2948
E025 | train 1.8037 | val 2.2183
E050 | train 1.4578 | val 2.5338
E075 | train 1.0373 | val 3.0025
E100 | train 0.6403 | val 3.5997
E125 | train 0.3434 | val 4.3531
E150 | train 0.1654 | val 5.3254
E175 | train 0.1013 | val 6.6338
E200 | train 0.0416 | val 7.2134
✅ Model saved
Generated slide shape: torch.Size([50, 10])
✅ Slide JSON saved: generated_slide_with_color.json


# For seperate interpolation

In [5]:
import torch
from pathlib import Path
import json
import random

# Assuming these are defined elsewhere in your code
# SLIDE_WIDTH_EMU = some_integer_value  # e.g., 9144000 (PowerPoint EMU units for slide width)
# SLIDE_HEIGHT_EMU = some_integer_value # e.g., 6858000 (PowerPoint EMU units for slide height)
# std = list_of_standard_deviations     # e.g., [std_x, std_y, std_w, std_h]
# mu = list_of_means                    # e.g., [mu_x, mu_y, mu_w, mu_h]
# K = number_of_context_slides          # e.g., 3
# PAD_VALUE = some_padding_value        # e.g., -1

def compute_average_color(emb, msk, device='cuda'):
    """Compute the average color of an embedding tensor, ensuring device consistency."""
    emb = emb.to(device)
    msk = msk.to(device)
    valid = msk == 1
    if valid.sum() == 0:
        return torch.tensor([0.0, 0.0, 0.0, 0.0], device=device)  # RGBA
    colors = emb[valid, 6:]  # Assuming color in dimensions 6-9 (RGBA)
    avg_color = colors.mean(dim=0)
    return avg_color

@torch.no_grad()
def generate_next_slide(model, ctx_emb, ctx_m, ctx_p, all_mats, layout_labels, device='cuda'):
    """Generate the next slide, ensuring all tensors are on the specified device."""
    model.eval()
    ctx_emb = ctx_emb.to(device)
    ctx_m = ctx_m.to(device)
    ctx_p = torch.tensor(ctx_p, dtype=torch.long).to(device).unsqueeze(0)  # (1, K)
    p_layout = model(ctx_emb.unsqueeze(0), ctx_m.unsqueeze(0), ctx_p)
    layout_type = torch.argmax(p_layout, dim=-1).item()
    candidates = [(m, i) for i, (m, l) in enumerate(zip(all_mats, layout_labels)) if l == layout_type]

    if not candidates:
        gen = all_mats[0].to(device)
    else:
        # Compute average color of context slides
        ctx_avg_colors = []
        for k in range(K):
            emb = ctx_emb[k]  # (50, 10)
            msk = ctx_m[k]    # (50,)
            avg_color = compute_average_color(emb, msk, device=device)
            ctx_avg_colors.append(avg_color)
        ctx_avg_color = torch.stack(ctx_avg_colors).mean(dim=0)

        # Select candidate with closest average color
        min_dist = float('inf')
        best_candidate = None
        for cand_emb, idx in candidates:
            cand_emb = cand_emb.to(device)
            cand_msk = (cand_emb[:, :6] != PAD_VALUE).any(-1).float().to(device)
            cand_avg_color = compute_average_color(cand_emb, cand_msk, device=device)
            dist = torch.norm(cand_avg_color - ctx_avg_color)
            if dist < min_dist:
                min_dist = dist
                best_candidate = cand_emb
        gen = best_candidate if best_candidate is not None else random.choice(candidates)[0].to(device)

    # Denormalize and clamp geometry
    slide_width_emu = torch.tensor(SLIDE_WIDTH_EMU, device=device, dtype=torch.float32)
    slide_height_emu = torch.tensor(SLIDE_HEIGHT_EMU, device=device, dtype=torch.float32)
    gen_denorm = gen[:, :4] * torch.tensor(std[:4], device=device) + torch.tensor(mu[:4], device=device)

    # Clamp x and y coordinates (scalars for min and max)
    gen_denorm[:, 0] = torch.clamp(gen_denorm[:, 0], 0, SLIDE_WIDTH_EMU)
    gen_denorm[:, 1] = torch.clamp(gen_denorm[:, 1], 0, SLIDE_HEIGHT_EMU)

    # Clamp width and height using torch.max and torch.min
    zero_tensor = torch.tensor(0.0, device=device, dtype=torch.float32)
    gen_denorm[:, 2] = torch.max(gen_denorm[:, 2], zero_tensor)  # Ensure width >= 0
    gen_denorm[:, 2] = torch.min(gen_denorm[:, 2], slide_width_emu - gen_denorm[:, 0])  # Ensure width <= slide_width - x
    gen_denorm[:, 3] = torch.max(gen_denorm[:, 3], zero_tensor)  # Ensure height >= 0
    gen_denorm[:, 3] = torch.min(gen_denorm[:, 3], slide_height_emu - gen_denorm[:, 1])  # Ensure height <= slide_height - y

    # Renormalize the clamped geometry
    gen[:, :4] = (gen_denorm - torch.tensor(mu[:4], device=device)) / torch.tensor(std[:4], device=device)

    return gen

# Example usage (ensure val_ds_raw, all_mats, layout_labels, and model are defined)
ctx_emb = torch.stack(val_ds_raw[0][:K])  # (K, 50, 10)
ctx_m = torch.stack(val_ds_raw[1][:K])    # (K, 50)
ctx_p = val_ds_raw[2][:K]                 # List of K pres_ids
gen = generate_next_slide(model, ctx_emb, ctx_m, ctx_p, all_mats, layout_labels, device='cuda')
print("Generated slide shape:", gen.shape)

out_path = Path('generated_slide_with_color.json')
json.dump({"generated": gen.cpu().tolist()}, out_path.open("w"), indent=2)
print("✅ Slide JSON saved:", out_path)

Generated slide shape: torch.Size([50, 10])
✅ Slide JSON saved: generated_slide_with_color.json
